# v8.5 / v8.6 past-L2 re-test — is the decode residual LATENCY or OCCUPANCY?

**Not a new kernel — a measurement** (mirrors `v9_task1_regime.ipynb`). The v9 close-out flagged that the
v8.5 (double-buffer) and v8.6 (occupancy / ILP) **nulls were measured only at L2-resident sizes** (N_k <= 16K)
— the exact confound Task 1 was built to kill. Task 1 then showed real headroom **past L2** (achieved %HBM
caps ~28% at high occupancy vs a ~70% ceiling), where load-latency-hiding *could* finally bite.

This notebook re-runs the v8.5/v8.6 family through the clock-locked, L2-flushed `bench.regime` sweep **PAST
L2**, against the Cut-1 baseline they fork (`v8_gqa`) and the relayout that won (`v8_gqa_ss` = v8.7). The
decisive read is **%HBM vs N_k per backend**:
- a latency-hiding arm (`db`/`occ`/`ilp`) **rises above Cut-1 past L2** -> the residual is **load latency**
  -> *"decode-schedule CLOSED" reopens* and the next lever is pipelining, not the score-stationary relayout alone;
- all arms **track Cut-1 flat** -> confirmed **dead ends even past L2**; the floor is the serial
  online-softmax recurrence (only the v8.7 relayout removed it), and *CLOSED stands*, now confound-free.

**Prediction (record before the run — the v8.6 counter-prediction):** TLP (occ) and ILP can't hide a
*serial per-row recurrence*, so `db`/`occ`/`ilp` stay **NULL vs Cut-1 even past L2**; only `ss` (which
*removed* the recurrence) sits higher. **Counter-prediction (the reopener):** if the residual is HBM *load*
latency, `db` (overlaps the KV load) lifts %HBM above Cut-1 once N_k spills L2. Either is first-class.

**Needs a root T4** to lock clocks (vast.ai ~$0.10-0.20/hr) for the publishable wall-times; on free Colab the
counter-free %HBM/eff_bw signal still survives (intra-run, clock-robust), only cross-run wall-times don't.

## 0. Dependencies + GPU (venv-safe; matplotlib for the plot)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU on this runtime. FIX: pick a T4 (Colab) or a root/bare-metal T4 (vast.ai).')

# matplotlib for the decisive plot; numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit('A GPU is present but torch was CPU-only -- installed CUDA build. Restart + re-run.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Lock clocks (root only — loud warning + continue if not)

In [ ]:
from bench.regime import lock_clocks
ok, sm_mhz, mem_mhz, throttle = lock_clocks()
print('locked =', ok, '| sm =', sm_mhz, 'MHz | mem =', mem_mhz, 'MHz | throttle:', throttle or 'none')
if not ok:
    print()
    print('>>> Running UNLOCKED: trust %HBM / eff_bw (intra-run, clock-robust);')
    print('>>> do NOT compare absolute us/tok across runs. For the publishable verdict, use a root T4.')

## 3. Roofline framing — identical FP16 floor for all five backends (a pure SCHEDULE test)

In [ ]:
# All five backends are FP16-in (b=2) GQA M-packing kernels, so the roofline is IDENTICAL: decode
# AI = 2/b, HBM-bound, same floor. The model is BLIND to the SCHEDULE -- which is the entire variable here
# (load-overlap / occupancy / ILP / score-stationary relayout). So this is a pure prediction-vs-measured
# SCHEDULE test: does any latency-hiding lever lift achieved %HBM past L2, where Task 1 showed ~28%->70% headroom?
from roofline.archs import get_arch
arch = get_arch('sm_75')
N_CROSS = int(arch.l2_mb * 1e6 / (2 * 128 * 2))   # KV working set 2*N_k*d*b = 4 MB, B=1 H_kv=1 d=128 fp16
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s | L2', arch.l2_mb, 'MB')
print('L2 crossing (B=1 H_kv=1 d=128 fp16): N_k ~=', N_CROSS, '(H_kv=8 crosses 8x earlier)')
print()
print('PREDICTION: db/occ/ilp NULL vs Cut-1 (v8_gqa) even past L2 (serial recurrence unhideable);')
print('only v8_gqa_ss (score-stationary, recurrence REMOVED) sits higher.')
print('COUNTER: db lifts %HBM above Cut-1 past L2 => residual is load latency => "CLOSED" reopens.')

## 4. Build the v8.5 / v8.6 family + the two references (Cut-1 baseline + v8.7 winner)

In [ ]:
import glob, os, shutil
from bindings.load import build_kernel
# v8_gqa   = Cut 1 (CUDA-core M-pack GEMV) -- the baseline db/occ/ilp all fork.
# v8_gqa_db/occ/ilp = the latency-hiding arms (v8.5 double-buffer, v8.6 occupancy, v8.6 ILP).
# v8_gqa_ss = v8.7 score-stationary RELAYOUT (removed the per-key reduction) -- the reference that won.
BACKENDS = ['v8_gqa', 'v8_gqa_ss', 'v8_gqa_db', 'v8_gqa_occ', 'v8_gqa_ilp']
for name in BACKENDS:
    for d in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/fa_{name}')):
        if not glob.glob(os.path.join(d, '*.so')):
            shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
mods = {name: build_kernel(name) for name in BACKENDS}
for name in BACKENDS:
    print(f'built {name}:', mods[name] is not None)

## 5. THE PAST-L2 SWEEP — %HBM vs N_k, all five backends (L2-flushed)

In [ ]:
# B=1, H_kv in {1 (occupancy-starved), 8 (the 28% plateau)}, d=128, N_k 1K..128K, gqa_group=1 -- so
# this is DIRECTLY comparable to the v9 Task 1 numbers (which also ran gqa_group=1). L2 FLUSHED each iter.
# One sweep per backend; collect structured rows for the plots.
from bench.regime import sweep
KV = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
rows = {}
for name in BACKENDS:
    print(f'================ {name} ================')
    rows[name] = sweep(name, KV, batches=[1], head_dims=[128], h_kvs=[1, 8])
    print()
print('collected', sum(len(r) for r in rows.values()), 'rows across', len(BACKENDS), 'backends')

## 6. THE DECISIVE PLOT — %HBM vs N_k per backend (does any arm lift past L2?)

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

COLORS = {'v8_gqa': 'tab:gray', 'v8_gqa_ss': 'tab:green',
          'v8_gqa_db': 'tab:blue', 'v8_gqa_occ': 'tab:orange', 'v8_gqa_ilp': 'tab:red'}
LABEL = {'v8_gqa': 'v8_gqa (Cut 1, baseline)', 'v8_gqa_ss': 'v8_gqa_ss (v8.7, ref)',
         'v8_gqa_db': 'v8_gqa_db (v8.5 double-buffer)', 'v8_gqa_occ': 'v8_gqa_occ (v8.6 occ)',
         'v8_gqa_ilp': 'v8_gqa_ilp (v8.6 ILP)'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, hk in zip(axes, (8, 1)):
    for name in BACKENDS:
        pts = sorted([r for r in rows[name] if r['H_kv'] == hk and r['d'] == 128], key=lambda r: r['N_k'])
        if not pts:
            continue
        ax.plot([r['N_k'] for r in pts], [r['hbm_pct'] for r in pts],
                marker='o', color=COLORS[name], label=(LABEL[name] if hk == 8 else None))
    ax.axhline(70, color='gray', ls=':', lw=1)                      # achievable ceiling
    ax.axvline(N_CROSS * (1 if hk == 1 else 1), color='black', ls=':', lw=1)  # d=128 H_kv=1 L2 crossing
    ax.set_xscale('log', base=2); ax.set_xlabel('N_k'); ax.set_title(f'H_kv={hk} (d=128)')
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_ylabel('% of peak HBM BW')
fig.legend(loc='lower center', ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.05))
fig.suptitle('v8.5/v8.6 past-L2 re-test: does any latency-hiding arm lift %HBM past L2? (dotted = L2 crossing / 70% ceiling)')
os.makedirs('docs/diagrams', exist_ok=True)
fig.savefig('docs/diagrams/v8_5_v8_6_pastL2.svg', bbox_inches='tight')
fig.savefig('docs/diagrams/v8_5_v8_6_pastL2.png', dpi=110, bbox_inches='tight')
print('saved docs/diagrams/v8_5_v8_6_pastL2.svg  (git add it to commit the figure)')
plt.show()

## 7. Past-L2 batch sweep — does occupancy (more blocks) change the verdict?

In [ ]:
# Fix N_k PAST L2 (32768, H_kv=1, d=128 -> 16 MB >> 4 MB L2), sweep B. The occ arm (4 blocks/SM) should
# help MOST where occupancy is the wall; if all arms stay flat, occupancy isn't the lever past L2 either.
from bench.regime import sweep
for name in BACKENDS:
    print(f'================ {name} (N_k=32768 past L2, batch sweep) ================')
    sweep(name, [32768], batches=[1, 8, 32, 64], head_dims=[128], h_kvs=[1], max_ws_gb=16.0)
    print()

## 8. Optional ncu cross-check (root only) — Cut-1 vs the double-buffer arm past L2

In [ ]:
# If clocks locked + ncu available, read DRAM% + L2-hit for Cut-1 vs the double-buffer arm at one past-L2
# shape. If db raises DRAM% above Cut-1, load-latency-hiding works past L2. Skips cleanly on ERR_NVGPUCTRPERM.
import subprocess, sys
METRICS = ('lts__t_sector_hit_rate.pct,'
           'dram__throughput.avg.pct_of_peak_sustained_elapsed,'
           'lts__throughput.avg.pct_of_peak_sustained_elapsed')
for name in ('v8_gqa', 'v8_gqa_db'):
    print(f'===== ncu {name} @ past-L2 N_k=65536 H_kv=1 d=128 =====')
    cmd = ['ncu', '--metrics', METRICS, '--launch-count', '5',
           '--kernel-name', 'regex:(gqa|db)', '--target-processes', 'all',
           sys.executable, '-m', 'bench.regime', '--profile', '1,1,65536,128', '--backend', name]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        print(out.stdout[-2200:] if out.stdout else '(no stdout)')
        if 'ERR_NVGPUCTRPERM' in (out.stdout + out.stderr):
            print('>>> ncu blocked (ERR_NVGPUCTRPERM): need root. Counter-free %HBM above stands.')
    except FileNotFoundError:
        print('>>> ncu not installed; skipping (counter-free %HBM is primary).'); break
    except Exception as e:
        print('>>> ncu failed:', type(e).__name__, e)

## 9. Reset clocks

In [ ]:
from bench.regime import reset_clocks
reset_clocks()

## 10. Verdict (fill after the run)

Read the **%HBM vs N_k** plot (both H_kv panels) + the batch sweep:

| Outcome | Signature | Means |
|---|---|---|
| **Latency-bound (CLOSED reopens)** | a `db`/`occ`/`ilp` line rises **above `v8_gqa` (Cut-1)** past the L2 crossing | the residual decode limiter is **load latency**, hideable by pipelining -> "decode-schedule CLOSED" was an L2-resident artifact; double-buffer is back on the table (and the v9 FP8 "flips negative under flush" gets a latency explanation). Next lever: a deeper-pipeline v8.8. |
| **Occupancy-bound** | only `occ` rises, and only at H_kv=1 (the starved case), not H_kv=8 | the floor is occupancy at low grids; persistent-kernel / more-blocks is the lever, not pipelining. |
| **Confirmed dead ends (CLOSED stands)** | `db`/`occ`/`ilp` all **track Cut-1 flat**; only `ss` sits higher | the floor is the **serial online-softmax recurrence** -- only the score-stationary *relayout* removed it; TLP/ILP/load-overlap can't hide a serial dependency. "decode-schedule CLOSED" is now **confound-free**, and v9 FP8 = capacity+accuracy stands. v10 NVFP4 proceeds as planned. |

Then: fill `docs/results.md` Step 8.5/8.6 with the past-L2 result, **`git add docs/diagrams/v8_5_v8_6_pastL2.svg`**
(the matplotlib `savefig` runs on the host), and update the "decode-schedule CLOSED" wording + the v10 reopener
note accordingly.